# V03 - Autenticacao e ciclo de vida seguro de um space

**Objetivo:** autenticar no CDF e executar o ciclo completo de um recurso DMS temporario: criar, recuperar, inspecionar e excluir um `space`.

**Por que importa:** todo recurso DMS (container, view, instancia) vive dentro de um space. Dominar o ciclo de vida de um space e a base para qualquer operacao segura.

**Como funciona:** o notebook gera um ID aleatorio com prefixo `sp_ur_training_v03_` e, ao final, exclui apenas esse ID gerado nesta execucao.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v03',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Preparar identificador seguro
Um prefixo fixo + sufixo aleatorio garante que a limpeza so afete o recurso desta execucao.

In [ ]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import SpaceApply

# Confirma acesso sem revelar token ou secret. / Confirm access without revealing token or secret.
test_space = f'sp_ur_training_v03_{uuid4().hex[:8]}'
assert test_space.startswith('sp_ur_training_v03_')
print('space alvo desta execucao:', test_space)

## 2. Criar
Cria somente o namespace temporario desta execucao.

In [ ]:
created_space = client.data_modeling.spaces.apply(SpaceApply(
    space=test_space,
    name='UR training - V03',
    description='Temporary space created by the V03 onboarding lab; delete at the end of the run.',
))
created_space

## 3. Recuperar e inspecionar
Recupera o mesmo space e mostra as propriedades retornadas pelo SDK.

In [ ]:
retrieved_space = client.data_modeling.spaces.retrieve(test_space)
assert retrieved_space is not None, 'O space de teste nao foi recuperado.'

spaces_df = client.data_modeling.spaces.list(limit=100).to_pandas()
current_space_df = spaces_df.loc[spaces_df['space'].eq(test_space)]
current_space_df

## Mini-exercicio
- Antes de excluir, altere a `description` do space (reaplique `SpaceApply`) e recupere novamente para ver o campo atualizado.
- Liste quantos spaces do projeto comecam com `sp_` versus outros prefixos.

## 4. Limpeza idempotente
Exclui apenas o space criado acima; nunca use este padrao com um ID recebido externamente.

In [ ]:
assert test_space.startswith('sp_ur_training_v03_')
deleted_spaces = client.data_modeling.spaces.delete(test_space)
deleted_space = client.data_modeling.spaces.retrieve(test_space)
assert deleted_space is None
print('space_still_exists:', deleted_space is not None)
deleted_spaces